# NLP Project: Disaster Tweet Classification
**Student:** Priya  
**Course:** Data Science  
**Objective:** Classify a tweet as disaster-related (`1`) or not disaster-related (`0`) using natural language processing.

## Project workflow
1. Load and inspect the labelled tweet dataset.
2. Explore class balance and common terms.
3. Clean text and prepare labels.
4. Convert text to TF-IDF features.
5. Train and compare baseline text-classification models.
6. Evaluate the selected model on a held-out test set.
7. Save the trained pipeline for use in a small web app.

**Dataset note:** This notebook expects a labelled CSV such as `train.csv` with a tweet text column (usually `text`) and a target column (usually `target`). Put the file beside this notebook or upload it when prompted. Do not interpret example/demo output as real model performance.

## 1. Import libraries

In [ ]:
import re
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 120)
print("Libraries imported successfully.")

## 2. Load the dataset
Place `train.csv` in the same folder as this notebook. If your file has a different name, change `DATA_PATH` below. The code also supports a common alternate label column name.

In [ ]:
DATA_PATH = Path("train.csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    if candidates:
        print("CSV files found:", [p.name for p in candidates])
        DATA_PATH = candidates[0]
    else:
        raise FileNotFoundError(
            "No CSV file found. Download the labelled disaster-tweet dataset, "
            "save it as train.csv beside this notebook, and run this cell again."
        )

df = pd.read_csv(DATA_PATH)
print("Dataset file:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 3. Inspect structure and data quality

In [ ]:
print("Columns:", df.columns.tolist())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

TEXT_COL = "text" if "text" in df.columns else None
TARGET_COL = "target" if "target" in df.columns else None

if TEXT_COL is None:
    raise ValueError("Could not find a 'text' column. Update TEXT_COL to your tweet-text column.")
if TARGET_COL is None:
    possible = [c for c in df.columns if c.lower() in {"label", "class", "disaster", "is_disaster"}]
    if possible:
        TARGET_COL = possible[0]
    else:
        raise ValueError("Could not find a target/label column. Update TARGET_COL to your label column.")

df = df[[TEXT_COL, TARGET_COL]].copy()
df = df.dropna(subset=[TEXT_COL, TARGET_COL])
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(int)
df = df[df[TARGET_COL].isin([0, 1])].drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Label meanings: 0 = non-disaster, 1 = disaster-related")
display(df[TARGET_COL].value_counts().rename(index={0:"Non-disaster",1:"Disaster"}).to_frame("count"))

## 4. Explore class distribution

In [ ]:
class_counts = df[TARGET_COL].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Non-disaster (0)", "Disaster (1)"], [class_counts.get(0, 0), class_counts.get(1, 0)])
plt.title("Distribution of Tweet Classes")
plt.ylabel("Number of tweets")
plt.tight_layout()
plt.show()

print("Class proportions:")
display((df[TARGET_COL].value_counts(normalize=True).sort_index() * 100).round(2).rename(index={0:"Non-disaster (%)",1:"Disaster (%)"}).to_frame("percentage"))

## 5. Clean tweet text
We remove URLs, HTML-like tags, mentions, and punctuation while preserving letters and numbers. Hashtags are retained as words by removing only the `#` symbol.

In [ ]:
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("#", "")
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df[TEXT_COL].apply(clean_tweet)
display(df[[TEXT_COL, "clean_text", TARGET_COL]].head(10))

## 6. Examine frequent words
This is a simple exploratory view of individual words after cleaning. It is not a substitute for understanding context; words such as “fire” may be used figuratively.

In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def top_words(series, n=15):
    words = []
    for text in series:
        words.extend(w for w in text.split() if w not in ENGLISH_STOP_WORDS and len(w) > 2)
    return Counter(words).most_common(n)

for label, label_name in [(0, "Non-disaster"), (1, "Disaster")]:
    counts = top_words(df.loc[df[TARGET_COL] == label, "clean_text"])
    if counts:
        terms, freqs = zip(*counts)
        plt.figure(figsize=(9, 4))
        plt.barh(list(terms)[::-1], list(freqs)[::-1])
        plt.title(f"Most frequent words: {label_name}")
        plt.xlabel("Frequency")
        plt.tight_layout()
        plt.show()

## 7. Prepare train and test data
A stratified split keeps the class proportions similar in both subsets. TF-IDF is fitted inside each model pipeline to avoid learning vocabulary from the test set.

In [ ]:
X = df["clean_text"]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")

## 8. Train and compare models
We compare Logistic Regression and Multinomial Naive Bayes as interpretable baseline models for TF-IDF text features. The cross-validation score is calculated on the training split only.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
        ("classifier", MultinomialNB(alpha=0.5))
    ])
}

comparison = []
for name, model in models.items():
    model.fit(X_train, y_train)
    cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring="f1")
    pred = model.predict(X_test)
    comparison.append({
        "Model": name,
        "CV F1 mean": cv_scores.mean(),
        "Test accuracy": accuracy_score(y_test, pred),
        "Test precision": precision_score(y_test, pred, zero_division=0),
        "Test recall": recall_score(y_test, pred, zero_division=0),
        "Test F1": f1_score(y_test, pred, zero_division=0)
    })

results = pd.DataFrame(comparison).sort_values("Test F1", ascending=False).reset_index(drop=True)
display(results.round(4))
best_name = results.loc[0, "Model"]
best_model = models[best_name]
print("Selected model based on held-out test F1:", best_name)

## 9. Evaluate the selected model

In [ ]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Non-disaster", "Disaster"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Non-disaster", "Disaster"], cmap=None
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

if hasattr(best_model, "predict_proba") and y_test.nunique() == 2:
    RocCurveDisplay.from_estimator(best_model, X_test, y_test)
    plt.title(f"ROC Curve — {best_name}")
    plt.tight_layout()
    plt.show()

    PrecisionRecallDisplay.from_estimator(best_model, X_test, y_test)
    plt.title(f"Precision–Recall Curve — {best_name}")
    plt.tight_layout()
    plt.show()

## 10. Try the model on your own examples

In [ ]:
examples = [
    "Massive flooding has damaged several homes and roads in the area.",
    "That exam was a disaster, I completely forgot the answers.",
    "Emergency teams are helping people after the earthquake."
]
example_predictions = best_model.predict([clean_tweet(t) for t in examples])
for tweet, label in zip(examples, example_predictions):
    print(f"Tweet: {tweet}\nPrediction: {'Disaster-related' if label == 1 else 'Not disaster-related'}\n")

## 11. Save the model for the web app
The saved pipeline includes both TF-IDF vectorization and the classifier, so the app can accept raw tweet text and apply the same transformations.

In [ ]:
MODEL_PATH = Path("disaster_tweet_model.joblib")
joblib.dump(best_model, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH.resolve()}")

## 12. Conclusion and limitations

This project follows an NLP workflow to classify tweets into disaster-related and non-disaster-related categories. It explores the labelled dataset, cleans tweet text, converts language into TF-IDF features, compares two baseline classifiers, evaluates the selected model, and saves it for a web interface.

**Important limitations**
- Results depend on the dataset and its label quality; run all cells with the actual dataset before reporting metrics.
- A tweet classifier can make mistakes because words can be figurative, sarcastic, or context-dependent.
- This prototype is an educational aid, not a replacement for official emergency monitoring or human verification.
- Model performance should be reviewed on fresh, representative data before any real-world use.